# Multi-seed confidence intervals — WiDS ICU mortality

Five seeds per split regime, so the headline claims carry intervals and a
paired test rather than resting on one draw. Three claims need this:

1. the federated hybrid beats APACHE IVa (+0.059 AUROC within-site);
2. federation costs 0.021 AUROC within-site against centralized LightGBM;
3. the federated model degrades least across unseen hospitals (−0.0012 against
   −0.0068 centralized) — the smallest gap of the three, and the one most
   likely to be inside seed noise.

Attach `fedlgbm-src` and the **WiDS Datathon 2020** competition.
Run with **Save Version → Save & Run All (Commit)**.

In [ ]:
import sys, time, json, shutil
from pathlib import Path

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/outputs')

hits = sorted(INPUT.glob('*/**/fedlgbm/__init__.py'))
assert hits, 'fedlgbm not found — attach the fedlgbm-src dataset'
sys.path.insert(0, str(hits[0].parent.parent))

train_files = sorted(INPUT.glob('**/training_v2.csv'))
assert train_files, 'attach the WiDS Datathon 2020 competition'
DATA = train_files[0].parent
print('source:', hits[0].parent.parent)
print('data:', train_files[0])

import torch
DEVICE = 'cpu'
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'gpu: {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}')
    try:
        (torch.zeros(8, 8, device='cuda') @ torch.ones(8, 8, device='cuda')).sum().item()
        DEVICE = 'cuda'
        print('CUDA preflight passed')
    except Exception as exc:
        print(f'*** CUDA unusable ({type(exc).__name__}) — using CPU ***')
# All seeds must share one device: results are not bit-identical across
# devices, and a device shift inside an aggregate would sit in the interval.
print('device =', DEVICE)

In [ ]:
from fedlgbm import ExperimentConfig, setup_logging
from fedlgbm.ablations import run_multi_seed

setup_logging()

def build(split_mode: str, name: str) -> ExperimentConfig:
    cfg = ExperimentConfig()
    cfg.name = name
    cfg.output_dir = OUT
    cfg.device = DEVICE
    cfg.skip_privacy_attacks = True   # measured separately; costs a decoder per seed

    cfg.data.domain = 'wids'
    cfg.data.data_dir = DATA
    cfg.data.file_glob = 'training_v2.csv'
    cfg.data.max_rows = None
    cfg.data.max_per_class = None
    cfg.data.split_mode = split_mode

    cfg.partition.scheme = 'natural'
    cfg.partition.min_samples_per_client = 200

    cfg.fed.n_rounds = 30
    cfg.fed.local_epochs = 2
    cfg.mlp.hidden_dims = (256, 128)
    cfg.mlp.embedding_dim = 64

    cfg.lgbm.class_weight = None
    cfg.lgbm.num_leaves = 255
    cfg.lgbm.n_estimators = 1000
    return cfg

SEEDS = (42, 43, 44, 45, 46)
# stacked_mlp_lgbm is the most expensive model by far (~350s/seed) and is not
# load-bearing for any claim under test, so it is dropped to keep this affordable.
SKIP = {'lstm', 'isolation_forest', 'stacked_mlp_lgbm', 'centralized_mlp_lgbm'}
# The reference must be a model that runs — pairing against a skipped model
# yields a summary with no significance test, discovered only at the end.
REFERENCE = 'fed_hybrid_mlp_lgbm'
print('seeds:', SEEDS, '| reference:', REFERENCE)

In [ ]:
summaries = {}
t0 = time.time()
for mode, name in (('random', 'wids_ms_within_site'),
                   ('site_holdout', 'wids_ms_cross_site')):
    print('=' * 70, f'\n{name}  (split_mode={mode})\n', '=' * 70)
    summaries[name] = run_multi_seed(
        build(mode, name), seeds=SEEDS, skip=SKIP, reference=REFERENCE
    )
print(f'\ntotal {(time.time() - t0) / 60:.1f} min')

In [ ]:
for name, s in summaries.items():
    print('\n' + '=' * 70, f'\n{name}   n_seeds={s["aggregated"]["n_seeds"]}\n', '=' * 70)
    for model, m in sorted(s['aggregated']['models'].items()):
        auc = m.get('roc_auc')
        ap = m.get('average_precision')
        if not auc:
            continue
        print(f"{model:42s} AUROC {auc['mean']:.4f} "
              f"[{auc['ci95_low']:.4f}, {auc['ci95_high']:.4f}]   "
              f"AUPRC {ap['mean']:.4f} [{ap['ci95_low']:.4f}, {ap['ci95_high']:.4f}]")

    print('\n--- paired vs', s.get('reference'), '(accuracy) ---')
    print(json.dumps(s['significance_accuracy'], indent=2)[:1500])

for name in summaries:
    for fname in ('multi_seed_summary.json', 'multi_seed_table.md'):
        src = OUT / name / fname
        if src.exists():
            shutil.copy2(src, Path('/kaggle/working') / f'{name}_{fname}')

shutil.make_archive('/kaggle/working/wids_multiseed_outputs', 'zip', OUT)
print('\narchived')